# RustyStats performance repro: freMTPL2 50k ridge fit

This notebook isolates the RustyStats model fit used by the destyler French MTPL report.

- `fremtpl2_50k_rustystats_train.parquet` contains the exact training frame passed to `rustystats.glm_dict`; `ClaimCount` has been replaced by the CatBoost teacher prediction because destyler fits the student to the teacher.
- `glm_kwargs.json` contains the direct `rustystats.glm_dict(...)` keyword arguments, including the same terms, interactions, offset, and deterministic input transforms.
- `reference_student.rsglm` is the fitted model from the successful timed run.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from time import perf_counter

import numpy as np
import polars as pl
import rustystats as rs

ROOT = Path.cwd()
if not (ROOT / "glm_kwargs.json").exists():
    ROOT = Path("rustystats-performance")
print("root:", ROOT.resolve())

In [ ]:
config = json.loads((ROOT / "fit_config.json").read_text())
glm_kwargs = json.loads((ROOT / "glm_kwargs.json").read_text())
train = pl.read_parquet(ROOT / config["train_parquet"])

print("train shape:", train.shape)
print("response mean (teacher target):", float(train[config["response"]].mean()))
print("terms:", len(glm_kwargs["terms"]))
print("interactions:", len(glm_kwargs.get("interactions") or []))
print("input transforms:", len(glm_kwargs.get("input_transforms") or []))
print(
    "fit config:", {k: config[k] for k in ["regularization", "cv", "selection", "n_alphas", "seed"]}
)

In [ ]:
print("Term names:")
for name, spec in glm_kwargs["terms"].items():
    print(f"  {name}: {spec}")

print("\nInteraction specs:")
for spec in glm_kwargs.get("interactions") or []:
    print(" ", spec)

In [ ]:
def timed(label, fn):
    start = perf_counter()
    try:
        return fn()
    finally:
        print(f"{label}: {perf_counter() - start:.3f}s")


builder = timed(
    "glm_dict builder construction",
    lambda: rs.glm_dict(**glm_kwargs, data=train, seed=config["seed"]),
)

## Optional baseline: unregularized fit

On the timed run this failed quickly with a singular design matrix. It is useful as a smoke test because it separates linear algebra singularity from the slower regularized CV path.


In [ ]:
try:
    unregularized = timed("unregularized fit", lambda: builder.fit())
    print("unregularized params:", len(unregularized.params))
except Exception as exc:
    print(type(exc).__name__, exc)

## Reproduce the successful fitted model

This is the slow stage from destyler timing: ridge regularization with 2-fold CV and 20 alphas.


In [ ]:
fit_kwargs = dict(
    regularization=config["regularization"],
    cv=config["cv"],
    selection=config["selection"],
    n_alphas=config["n_alphas"],
    cv_seed=config["seed"],
)
model = timed("ridge CV fit", lambda: builder.fit(**fit_kwargs))
print("params:", len(model.params))
print(
    "input transforms:",
    len(
        model.input_transforms if not callable(model.input_transforms) else model.input_transforms()
    ),
)

In [ ]:
reference_path = ROOT / config["reference_student"]
if reference_path.exists():
    ref = rs.GLMModel.from_bytes(reference_path.read_bytes())
    sample = train.head(5_000)
    pred = np.asarray(model.predict(sample), dtype=float)
    ref_pred = np.asarray(ref.predict(sample), dtype=float)
    print("max abs prediction delta:", float(np.max(np.abs(pred - ref_pred))))
    print("mean abs prediction delta:", float(np.mean(np.abs(pred - ref_pred))))
else:
    print("No reference model found")

In [ ]:
out = ROOT / "reproduced_student.rsglm"
out.write_bytes(model.to_bytes())
print("wrote", out, out.stat().st_size, "bytes")